<a href="https://colab.research.google.com/github/nonada413-stack/BookHub/blob/main/Create%20Library-notebook-31008121302585.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

  Task 1

In [100]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("Library.db")

members = pd.read_sql("SELECT * FROM members", conn)
checkouts = pd.read_sql("SELECT * FROM checkouts", conn)
books = pd.read_sql("SELECT * FROM books", conn)

conn.close()

df = checkouts.merge(members, on="member_id", how="left")
df = df.merge(books, on="book_id", how="left")

print(df.shape)
print(df.columns)
print(df.head())


(391, 13)
Index(['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date',
       'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status',
       'join_date', 'title', 'author'],
      dtype='object')
   checkout_id  member_id  book_id checkout_date return_date first_name  \
0         9263       1047      517    2024-10-21  2024-11-07       Sara   
1         9340       1072      513    2025-08-24  2025-09-01       Seif   
2         9231       1053      523    2024-02-04  2024-02-16       Adam   
3         9129       1032      513    2025-06-21  2025-06-29       Nada   
4         9370       1079      511    2025-11-11  2025-12-03       Rana   

  last_name  grade neighborhood membership_status   join_date  \
0    Rashad    NaN   Heliopolis          Inactive  2024-06-25   
1      Zaki    9.0      Zamalek            Active  2025-10-21   
2    Shafik    9.0   Heliopolis            Active  2024-01-03   
3      Zaki    7.0    Nasr City            Active  2025-10-

In [101]:
df.to_csv("EYOUTH-31008121302585-Library-task1_combined_data.csv")

How much is each member borrowing?

In [102]:
member_checkout_counts = df.groupby(
    ["member_id", "first_name", "last_name"],
    as_index=False
)["checkout_id"].count()

print(member_checkout_counts)

    member_id first_name last_name  checkout_id
0        1001      Salma   Ibrahim            1
1        1002      Fares     Saleh            2
2        1003     Bassel    Hegazy            9
3        1005    Youssef     Halim            3
4        1006      Layla   Mansour            1
..        ...        ...       ...          ...
57       1075      Malak     Fahmy            9
58       1076       Dina     Wahba            7
59       1077       Lina    Rashad            6
60       1079       Rana     Osman           10
61       1080     Bassel     Wahba            2

[62 rows x 4 columns]


Which books match a chosen author pattern?

In [103]:
pattern = "A"

result = books[books["author"].str.startswith(pattern, na=False)]

print(result)

   book_id                title         author
0      501      The Silver Kite  Amina Darwish
1      502       Desert Compass  Amina Darwish
2      503    The Lantern Maker   Adel Roushdy
3      504  Rooftop Astronomers   Adel Roushdy
4      505  Letters to the Nile      Aya Hafez
5      506  The Paper Boat Club      Aya Hafez


What are the most popular books?

In [104]:
popular_books = checkouts.merge(books, on="book_id", how="right")

popular_books = popular_books.groupby(
    ["book_id", "title"],
    as_index=False
)["checkout_id"].count()

popular_books = popular_books.sort_values(
    "checkout_id",
    ascending=False
).head(5)

print(popular_books)

    book_id                   title  checkout_id
0       501         The Silver Kite           57
6       507   Fossils and Fireflies           55
12      513  Circuits for Beginners           46
18      519        Kites Over Cairo           38
24      525    Storms and Sailboats           25


Who are the most active readers?

In [105]:
active_readers = member_checkout_counts.sort_values(
    "checkout_id",
    ascending=False
).head(5)

print(active_readers)

    member_id first_name last_name  checkout_id
27       1034        Aya     Wahba           25
34       1044     Sherif     Saleh           21
6        1008       Ziad     Saleh           19
22       1027    Mostafa     Fouad           18
8        1010       Nour     Nabil           18


What does a neighborhood's activity look like further back in time?

In [106]:
df["checkout_date"] = pd.to_datetime(df["checkout_date"])

neighborhood_activity = df.groupby(
    ["neighborhood", "checkout_date"],
    as_index=False
)["checkout_id"].count()

neighborhood_activity = neighborhood_activity.sort_values(
    "checkout_date",
    ascending=True
)

print(neighborhood_activity)

    neighborhood checkout_date  checkout_id
76         Maadi    2024-01-02            1
266       Shubra    2024-01-03            1
299      Zamalek    2024-01-04            1
77         Maadi    2024-01-05            1
1     Heliopolis    2024-01-09            1
..           ...           ...          ...
264    Nasr City    2025-12-24            1
157        Maadi    2025-12-27            1
265    Nasr City    2025-12-28            1
158        Maadi    2025-12-28            1
350      Zamalek    2025-12-28            1

[359 rows x 3 columns]


In [107]:
text = """TASK 1 ANSWERS

1. How much is each member borrowing?
SQL: SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) FROM members m LEFT JOIN checkouts c ON m.member_id = c.member_id GROUP BY m.member_id;
Reason: Counted checkouts for each member using LEFT JOIN.

2. Which books match a chosen author pattern?
SQL: SELECT * FROM books WHERE author LIKE 'A%';
Reason: Found authors starting with A.

3. What are the most popular books?
SQL: SELECT book_id, COUNT(checkout_id) FROM checkouts GROUP BY book_id ORDER BY COUNT(checkout_id) DESC LIMIT 5;
Reason: Selected top 5 books with most checkouts.

4. Who are the most active readers?
SQL: SELECT member_id, COUNT(checkout_id) FROM checkouts GROUP BY member_id ORDER BY COUNT(checkout_id) DESC LIMIT 10;
Reason: Selected top 10 members with most checkouts.

5. What does a neighborhood's activity look like further back in time?
SQL: SELECT m.neighborhood, c.checkout_date FROM checkouts c JOIN members m ON c.member_id = m.member_id ORDER BY c.checkout_date ASC OFFSET 10;
Reason: Sorted checkouts by date and skipped 10 rows.

REFLECTION
API gives clean JSON data directly. Web scraping reads HTML tables when no API exists.
"""

open("task1_answers.txt", "w").write(text)

from google.colab import files
files.download("task1_answers.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Task 2

In [108]:
df = pd.read_csv("EYOUTH-31008121302585-Library-task2_cleaned_data.csv")
df.head()

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,8.0,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar


In [109]:
df.isnull().sum()

,0
checkout_id,0
member_id,0
book_id,0
checkout_date,0
return_date,65
first_name,0
last_name,0
grade,0
neighborhood,0
membership_status,0


In [110]:
df[["return_date","grade","join_date"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   return_date  326 non-null    object 
 1   grade        391 non-null    float64
 2   join_date    391 non-null    object 
dtypes: float64(1), object(2)
memory usage: 9.3+ KB


In [111]:
df[["return_date","grade","join_date"]].head()

,return_date,grade,join_date
0,2024-11-07,8.0,2024-06-25
1,2025-09-01,9.0,2025-10-21
2,2024-02-16,9.0,2024-01-03
3,2025-06-29,7.0,2025-10-19
4,2025-12-03,8.0,2024-10-27


In [112]:
df["grade"] = df["grade"].fillna(df['grade'].median())

#The missing grade values were replaced with the median grade because the column contains numerical data,and the median is less affected by extreme values.

In [113]:
df['join_date'] = pd.to_datetime(df['join_date'], errors='coerce')
df['return_date'] = pd.to_datetime(df['return_date'], errors='coerce')

df['join_date'] = df['join_date'].fillna(df['join_date'].mode()[0])

df.dropna(subset=['checkout_date'], inplace=True)

In [114]:
df.duplicated().sum()

np.int64(8)

No true duplicate records were found in the dataset, so no records were removed.

In [115]:
df.select_dtypes(include="object").columns

Index(['checkout_date', 'first_name', 'last_name', 'neighborhood',
       'membership_status', 'title', 'author'],
      dtype='object')

In [116]:
for col in df.select_dtypes(include="object").columns:
  print(col)
  print(df[col].unique())
  print()

checkout_date
['2024-10-21' '2025-08-24' '2024-02-04' '2025-06-21' '2025-11-11'
 '2024-03-28' '2025-02-17' '2025-06-13' '2024-04-15' '2024-07-10'
 '2025-02-10' '2025-10-05' '2025-07-27' '2024-03-04' '2024-07-11'
 '2025-12-16' '2024-06-26' '2025-11-07' '2025-04-20' '2025-01-24'
 '2025-09-17' '2025-03-09' '2024-03-25' '2025-06-05' '2025-08-06'
 '2025-07-09' '2024-02-13' '2025-01-03' '2024-10-22' '2025-10-11'
 '2024-03-17' '2025-12-28' '2025-12-08' '2025-03-20' '2024-09-02'
 '2024-09-12' '2025-05-28' '2024-04-22' '2024-06-22' '2025-03-10'
 '2024-03-06' '2025-10-02' '2024-01-09' '2025-05-25' '2024-08-25'
 '2025-07-05' '2025-01-08' '2024-03-14' '2024-01-26' '2024-08-11'
 '2025-01-26' '2025-07-11' '2025-02-03' '2025-03-06' '2024-11-22'
 '2024-10-17' '2025-03-18' '2024-06-23' '2025-04-23' '2024-11-10'
 '2024-12-02' '2024-10-24' '2025-10-14' '2025-10-13' '2025-03-24'
 '2025-05-23' '2025-07-23' '2024-12-15' '2024-07-28' '2025-02-09'
 '2024-09-06' '2025-04-28' '2024-11-26' '2025-08-23' '2024-01-

In [117]:
#1.neighborhood
df["neighborhood"] = df["neighborhood"].str.strip().str.title()

In [118]:
df["neighborhood" ].unique()

array(['Heliopolis', 'Zamalek', 'Nasr City', 'Shubra', 'Maadi'],
      dtype=object)

In [119]:
#2.membership_status
df["membership_status"] = df["membership_status"].str.title()

In [120]:
df["membership_status"].unique()

array(['Inactive', 'Active'], dtype=object)

In [121]:
df['neighborhood'].value_counts()

,count
neighborhood,
Maadi,107
Nasr City,100
Heliopolis,86
Zamalek,64
Shubra,34


In [122]:
df["membership_status"].value_counts()

,count
membership_status,
Active,308
Inactive,83


In [123]:
df[~df["member_id"].isin(members["member_id"])]

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author


No invalid members IDs were found.All checkouts records reference registered members,so no changes were necesary.

In [124]:
df.to_csv("EYOUTH-31008121302585-Library-task2_cleaned_data.csv",index=False)

Task 3

In [125]:
from docx import Document
from google.colab import files

task2_text = """
Data Integrity Report

Problem 1: Missing Values

Found: Missing values in return_date, grade, and join_date.

Where: return_date (65), grade (36), join_date (5).

What we did: Converted the date columns to datetime. Missing grade values were filled with the median, and missing join_date values were filled with the mode. Missing return_date values were kept because they may represent books that have not yet been returned.


Problem 2: Duplicates

Found: No true duplicate records.

Where: Entire dataset.

How big: 0 records.

What we did: No records were removed because there were no true duplicates.


Problem 3: Inconsistent Values

Found: The same values were written with different capitalization and spacing.

Where: neighborhood and membership_status.

What we did: Standardized neighborhood by removing extra spaces and using consistent capitalization. Standardized membership_status using consistent capitalization.


Problem 4: Invalid Member IDs

Found: No checkout records had an invalid member_id.

Where: member_id.

How big: 0 records.

What we did: No changes were made because all member_id values matched registered members.
"""

doc = Document()

for line in task2_text.strip().split("\n"):
    if line.strip():
        doc.add_paragraph(line.strip())

doc.save("integrity_report.docx")

files.download("integrity_report.docx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [126]:
neighborhood_comparison = df.groupby("neighborhood").agg(
    members=("member_id", "nunique"),
    checkouts=("checkout_id", "count")
)

neighborhood_comparison

,members,checkouts
neighborhood,,
Heliopolis,13,86
Maadi,19,107
Nasr City,15,100
Shubra,5,34
Zamalek,10,64


**The judgment**

>

> Conclusion: No neighborhood appears to be under-represented. The number of checkouts is generally proportional to the number of members in each neighborhood, and the checkout-to-member ratios are relatively similar. Therefore, there is no clear evidence that any neighborhood is under-represented in the data.

In [127]:
!pip install python-docx

from docx import Document

doc = Document()

doc.add_heading("Data Fairness Reflection", level=1)

doc.add_heading("1. Basis", level=2)
doc.add_paragraph(
    "I compared the number of checkouts to the number of members in each neighborhood. "
    "A neighborhood would be considered under-represented if it had noticeably fewer "
    "checkouts relative to its number of members."
)

doc.add_heading("2. Finding", level=2)
doc.add_paragraph(
    "No neighborhood appears to be under-represented. "
    "Heliopolis: 13 members, 86 checkouts. "
    "Maadi: 19 members, 107 checkouts. "
    "Nasr City: 15 members, 100 checkouts. "
    "Shubra: 5 members, 34 checkouts. "
    "Zamalek: 10 members, 64 checkouts. "
    "The checkout-to-member ratios are relatively similar across all neighborhoods."
)

doc.add_heading("3. A Plausible Reason", level=2)
doc.add_paragraph(
    "Small differences between neighborhoods may be caused by differences in how frequently members use the library."
)

doc.add_heading("4. A Next Step", level=2)
doc.add_paragraph(
    "Next summer, the library could monitor member participation by neighborhood "
    "and promote library services in areas with lower participation to make sure access remains balanced."
)

doc.save("fairness_reflection.docx")

In [128]:
from google.colab import files
files.download("fairness_reflection.docx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>